### **Investigation on Dataset without Dividends**

In [2]:
import pandas as pd
import numpy as np

In [3]:
div_path = "/Users/utente/Desktop/Thesis/Thesis/src/49_Industry_Portfolios_Wout_Div.xlsx"
df = pd.read_excel(div_path)

rf_path = "/Users/utente/Desktop/Thesis/Thesis/src/Rf.xlsx"
rf = pd.read_excel(rf_path)

In [6]:
rf = rf[rf['Date'] >= '1969-07']
rf = rf[rf['Date'] <= '2021-12']
df = df[df['Date'] <= '2021-12']
#rf = rf.drop(rf.index[-1]) #exclude last observation to align the datasets
#rf = rf.iloc[:, [0, -1]] #keep risk-free rate column as it is the information we are interested in
print(df.shape)
print(rf.shape)

(630, 50)
(630, 5)


In [9]:
# Merge and obtain returns in excess of the risk-free rate

# 1. Merge on the 'Date' column
df = pd.merge(df, rf, on='Date', how='inner')

# 2. Identify all industry columns (49 columns).
#    We'll treat every column except 'Date' and 'RF' as an industry column.
industry_cols = [col for col in df.columns if col not in ['Date', 'RF']]

# 3. Subtract the risk-free rate from each industry column
for col in industry_cols:
    df[col] = df[col] - df['RF']

# 4. Drop RF column
df.drop(columns='RF', inplace=True)    

# 5. The final DataFrame 'merged_df' now has:
#    - Date
#    - 49 columns with monthly EXCESS returns
df.head()

,Date,Agric,Food,Soda,Beer,Smoke,Toys,Fun,Books,Hshld,...,Rtail,Meals,Banks,Insur,RlEst,Fin,Other,Mkt-RF,SMB,HML
0,1969-07,-9.45,-7.66,-4.07,-2.72,-5.40,-8.41,-10.73,-11.61,-4.03,...,-8.86,-10.52,-7.31,-10.94,-13.30,-9.74,-15.56,-7.53,-3.77,0.67
1,1969-08,3.61,3.71,4.86,5.83,2.38,5.75,11.05,8.95,4.06,...,7.39,8.46,12.12,7.23,14.47,8.05,7.88,4.18,0.43,-4.29
2,1969-09,0.97,-1.47,4.18,-2.66,3.33,1.31,-0.14,-1.35,-1.29,...,-1.44,-2.12,-4.11,6.31,-4.46,-3.34,-7.62,-3.60,0.56,-3.89
3,1969-10,15.73,7.16,6.23,10.91,11.69,5.67,12.88,6.98,5.74,...,5.47,8.40,9.93,4.82,3.92,9.50,2.99,4.46,3.31,-3.76
4,1969-11,-2.49,-1.50,-5.81,-3.58,-0.87,-8.99,-3.82,-4.77,-3.20,...,-4.95,-2.43,-9.55,-5.71,-6.83,-6.76,-2.77,-4.31,-3.10,-1.71


In [10]:
# Step 1: Separate the 'Date' column
dates = df['Date']
returns_data = df.loc[:, df.columns != 'Date']

# Step 2: Compute the mean of each portfolio
portfolio_means = returns_data.mean(axis=0)

# Step 3: Demean the returns
demeaned_returns = returns_data.subtract(portfolio_means, axis=1)

# Step 4: Verify the unconditional mean is zero
demeaned_means = demeaned_returns.mean(axis=0)
# Check that the absolute values are close to zero
print((demeaned_means.abs() < 1e-12).all())  # Should return True

# Step 5: Save back the returns dataset as "VWExcessRet"
VWExcessRet = demeaned_returns

VWExcessRet.columns = range(VWExcessRet.shape[1])

True


In [11]:
VWExcessRet

,0,1,2,3,4,5,6,7,8,9,...,42,43,44,45,46,47,48,49,50,51
0,-9.945476,-8.113476,-4.611762,-3.281333,-6.03873,-8.715778,-11.603746,-11.954127,-4.329825,-13.161143,...,-9.459683,-11.13054,-7.672349,-11.456825,-13.48819,-10.291175,-15.551381,-7.76254,-3.526143,0.776302
1,3.114524,3.256524,4.318238,5.268667,1.74127,5.444222,10.176254,8.605873,3.760175,1.228857,...,6.790317,7.84946,11.757651,6.713175,14.28181,7.498825,7.888619,3.94746,0.673857,-4.183698
2,0.474524,-1.923476,3.638238,-3.221333,2.69127,1.004222,-1.013746,-1.694127,-1.589825,-4.051143,...,-2.039683,-2.73054,-4.472349,5.793175,-4.64819,-3.891175,-7.611381,-3.83254,0.803857,-3.783698
3,15.234524,6.706524,5.688238,10.348667,11.05127,5.364222,12.006254,6.635873,5.440175,5.378857,...,4.870317,7.78946,9.567651,4.303175,3.73181,8.948825,2.998619,4.22746,3.553857,-3.653698
4,-2.985476,-1.953476,-6.351762,-4.141333,-1.50873,-9.295778,-4.693746,-5.114127,-3.499825,-4.081143,...,-5.549683,-3.04054,-9.912349,-6.226825,-7.01819,-7.311175,-2.761381,-4.54254,-2.856143,-1.603698
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
625,1.834524,-1.303476,-0.631762,-2.201333,2.83127,-10.605778,9.956254,0.035873,-0.329825,-1.211143,...,2.320317,-1.84054,1.517651,1.263175,3.85181,5.388825,1.988619,2.67746,-0.216143,0.006302
626,-4.645476,-2.823476,-7.071762,-4.011333,-9.10873,-10.335778,0.676254,0.755873,-4.479825,-9.781143,...,-5.209683,-1.35054,0.487651,-4.546825,-1.19819,-6.301175,-4.541381,-4.60254,0.913857,5.206302
627,2.294524,1.656524,4.908238,4.798667,-2.00873,3.274222,7.106254,1.615873,3.100175,11.708857,...,5.040317,0.26946,3.477651,9.983175,6.41181,10.818825,4.718619,6.41746,-2.126143,-0.343698
628,2.384524,-4.153476,-6.771762,-0.541333,-7.65873,-18.375778,-8.793746,-4.664127,0.560175,-0.441143,...,1.480317,-2.74054,-6.742349,-5.816825,-9.12819,-5.211175,-4.291381,-1.78254,-1.076143,-0.303698
